# Serif Takehome assignment

What is the list of machine readable file URLs that represent the Anthem PPO network in New York state?

Your output should be the list of machine readable file URLs corresponding to Anthem's PPO in New York state.

Here's the development workflow: 1. Write a rough draft with my thoughts as I go, 2. Make a cleaned up 'main' notebook from it, and 3. Produce a `main.py` script from the 'main' notebook.

This is the "draft" notebook.

The rough plan, as I see it:

1. Open the `json` and read it in **chunks** (or as stream). It's only 24G  uncompressed, but let's pretend this will have to extend to a 1TB file. 

2. [Refer to the published schema](https://github.com/CMSgov/price-transparency-guide/tree/master/schemas/table-of-contents) but anticipate getting something different. (This is why we do this in Jupyter!)

3. Handle multiply repeated data using Python's `set`. If performance is a concern, an XOR filter (or bloom filter; whatever a good library provides) is usually faster for checking set membership, at the risk (which can be made arbitrarily tiny) of false-positively rejecting a novel datapoint as new.

# Getting started

Let's get our imports out of the way. 

Read from`2026-02-01_anthem_index.json.gz`, chunking to avoid running out of memory. (It's ~11GB compressed and RAM is expensive)

In [7]:
# first, extract the json file 
!gzip -c -d 2026-02-01_anthem_index.json.gz > index.json

In [10]:
# Add dependencies as needed to the `pyproject.toml`.
!uv add pandas
!uv add json_stream
!uv add requests

Resolved 6 packages in 7ms
Audited 4 packages in 1ms
Resolved 8 packages in 499ms                                         
Prepared 2 packages in 135ms                                             
Installed 2 packages in 3ms                                 
 + json-stream==2.4.1
 + json-stream-rs-tokenizer==0.5.0
Resolved 13 packages in 154ms                                        
Prepared 5 packages in 197ms                                             
Installed 5 packages in 8ms                                 
 + certifi==2026.1.4
 + charset-normalizer==3.4.4
 + idna==3.11
 + requests==2.32.5
 + urllib3==2.6.3


In [12]:
import json
import os, sys
import pandas as pd
import json_stream
import httpx
import requests

---

# Chunking the JSON

I've only ever had to chunk tabular data before (raw Postgres dumps, CSVs, etc). 

See StackOverflow [10238340](https://stackoverflow.com/questions/10238340/), [7052947](https://stackoverflow.com/questions/7052947/), and [74858456](https://stackoverflow.com/questions/74858456/).

I love Pandas, but this is JSON and I don't know in advance if this will be nice, tabular data. Let's stay in JSON/dict land. `json_stream` looks like the way to go, especially with its [transient mode](https://pypi.org/project/json-stream/#:~:text=consume%20the%20data.-,Transient%20mode,-(default)) stream processing. (Visitor mode would probably be the most-correct way to do this- future improvement if I don't implement it that way).



In [17]:
# First let's look at the raw JSON
with open("index.json", "r") as ff:
    _sample = ff.read(1024)

_sample # it has linebreaks, nice

'{"reporting_entity_name":"Anthem Inc",\n"reporting_entity_type":"health insurance issuer",\n"last_updated_on":"2026-02-01",\n"version":"2.0.0",\n"reporting_structure":[\n{"reporting_plans":[{"plan_name":"LIVE HEALTH ONLINE - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"PRUDENT BUYER PPO - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"ENTPRS DIABETS - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"K HEALTH - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","pla

In [73]:
with open("index.json", "r") as ff:
    #_sample = ff.read(819200).split('\n')# looks like this doesn't repeat; the bulk is in the reporting plans
    _sample = ff.read(4096).split('\n')

_sample
# structure is unsurprising, is welcomly a list, with many small entries in `reporting_structure`.
#I can load one of these into memory at once as a large object


['{"reporting_entity_name":"Anthem Inc",',
 '"reporting_entity_type":"health insurance issuer",',
 '"last_updated_on":"2026-02-01",',
 '"version":"2.0.0",',
 '"reporting_structure":[',
 '{"reporting_plans":[{"plan_name":"LIVE HEALTH ONLINE - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"PRUDENT BUYER PPO - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"ENTPRS DIABETS - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":"208156960","plan_sponsor_name":"D & K MARQUEZ ENTERPRISES INC","plan_market_type":"group"},{"plan_name":"K HEALTH - D & K MARQUEZ ENTERPRISES INC - ANTHEM","plan_id_type":"EIN","issuer_name":"Anthem Inc","plan_id":

# Let's get the data out

The big thing is the `reporting_structure` part. This is not surprising. It has a list of `reporting_plans`, `in_network_files`, `allowed_amount_file`.

In [81]:
# Now let's see if json_stream reads this as expected without failing
# `with open` pattern fails (I/O operation on closed file) whoops
ff = open("index.json", "r")
index = json_stream.load(ff)

# This is a "read once" thing

#index['asdf'] #this parses the whole file before failing on a key error; not surprising

# now let's start investigating values
#index['reporting_entity_name']

#index['reporting_structure'][0]['reporting_plans'][0]['plan_name']

# <TransientStreamingJSONObject: TRANSIENT, STREAMING>, but I want to get this into a dict I can work with. It's small enough!
#_eg = index['reporting_structure'][0]

# need `json_stream.to_standard_types(_eg)`; too unwieldy

def tst( ts_json_objeect):
    # Utility function since I'll be using this a lot
    return json_stream.to_standard_types(ts_json_objeect)

tst(index['reporting_structure'][0])
#len(str(tst(index['reporting_structure'][0]))) # 153KB fits in memory very nicely!

{'reporting_plans': [{'plan_name': 'LIVE HEALTH ONLINE - D & K MARQUEZ ENTERPRISES INC - ANTHEM',
   'plan_id_type': 'EIN',
   'issuer_name': 'Anthem Inc',
   'plan_id': '208156960',
   'plan_sponsor_name': 'D & K MARQUEZ ENTERPRISES INC',
   'plan_market_type': 'group'},
  {'plan_name': 'PRUDENT BUYER PPO - D & K MARQUEZ ENTERPRISES INC - ANTHEM',
   'plan_id_type': 'EIN',
   'issuer_name': 'Anthem Inc',
   'plan_id': '208156960',
   'plan_sponsor_name': 'D & K MARQUEZ ENTERPRISES INC',
   'plan_market_type': 'group'},
  {'plan_name': 'ENTPRS DIABETS - D & K MARQUEZ ENTERPRISES INC - ANTHEM',
   'plan_id_type': 'EIN',
   'issuer_name': 'Anthem Inc',
   'plan_id': '208156960',
   'plan_sponsor_name': 'D & K MARQUEZ ENTERPRISES INC',
   'plan_market_type': 'group'},
  {'plan_name': 'K HEALTH - D & K MARQUEZ ENTERPRISES INC - ANTHEM',
   'plan_id_type': 'EIN',
   'issuer_name': 'Anthem Inc',
   'plan_id': '208156960',
   'plan_sponsor_name': 'D & K MARQUEZ ENTERPRISES INC',
   'plan_mark

# Now let's work with that data

We come into this with:


- `ff = open("index.json", "r"); index = json_stream.load(ff)` to re-open the file (you can only move forward!)
- `tst(ts_json_object)`: Utility function to get a proper object from a transient json stream

And we develop:

- `ein_lookup(ein)`: Int to dict (from json) for a given EIN
- ...

(When doing a rough draft in a notebook like this, I like to accumulate a description of what utility functions and bits of data I have.)

Let's get the in-network files from the first case. Just looking at the print-out above, there are standard queryparams here for authentication and expiration. I don't have control over what generates those keys, so let's keep them when we generate the URL list.


In [82]:
ff = open("index.json", "r")
index = json_stream.load(ff)
eg = tst(index['reporting_structure'][0])

In [88]:
# First, quick detour for EIN lookup
# network tools gives us https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/271050694.json
eg['reporting_plans']

def ein_lookup(ein):
    req = httpx.get(f"https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/{ein}.json")
    if req.status_code == 200:
        return json.loads(req.text)

ein_lookup(208156960) # it works!

{'lastupdated': '2026-02-01',
 'In-Network Negotiated Rates Files': [{'url': 'https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/CA_BCHSMED0000.json.gz',
   'displayname': 'CA_PPO_BCHSMED0000.json.gz'},
  {'url': 'https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/CA_ELHOMEDELHO.json.gz',
   'displayname': 'CA_ELHO_ELHS_ELHOMEDELHO.json.gz'},
  {'url': 'https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/CA_ELRKMEDELRK.json.gz',
   'displayname': 'CA_ENTPRS_DIABETS_ELRKMEDELRK.json.gz'},
  {'url': 'https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/CA_ENKHMEDENKH.json.gz',
   'displayname': 'CA_K_HLTH_ENKHMEDENKH.json.gz'},
  {'url': 'https://anthembcca.mrf.bcbs.com/2026-02_051_06B0_in-network-rates_1_of_5.json.gz?&Expires=1774274448&Signature=RaT2gJ9ehpnjr1FEtKhlSPWdrN4WWkUxHzqa-0YvgSCzfIFKa-uejwX4EEpWogKQ9zPMqGFqzNWacd8R1UMhTnTvnfKYzbp6DQjmrFkMsIpTVGovytxfwFediLt8xsgHrGQXNjk3CoGan8lIlThwRVyBQfKVv0We4OZ

In [ ]:
# ...